In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv('emails.csv')
display(df.head())

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [ ]:
df.tail()

,text,spam
5723,Subject: re : research and development charges...,0
5724,"Subject: re : receipts from visit jim , than...",0
5725,Subject: re : enron case study update wow ! a...,0
5726,"Subject: re : interest david , please , call...",0
5727,Subject: news : aurora 5 . 2 update aurora ve...,0


In [ ]:
df.shape

(5728, 2)

In [ ]:
df.isnull().sum()

,0
text,0
spam,0


In [ ]:
df.duplicated().sum()

np.int64(33)

In [ ]:
df=df.drop_duplicates()

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
print(f"Spam Message :{df[df['spam']==1].shape[0]}")
print(f"Ham Message :{df[df['spam']==0].shape[0]}")

Spam Message :1368
Ham Message :4327


In [ ]:
df.loc[:, "text"] = df["text"].str.lower()
df.head()

,text,spam
0,subject: naturally irresistible your corporate...,1
1,subject: the stock trading gunslinger fanny i...,1
2,subject: unbelievable new homes made easy im ...,1
3,subject: 4 color printing special request add...,1
4,"subject: do not have money , get software cds ...",1


In [ ]:
import regex
def clean_text(text):
    return re.sub(r"[^a-z\s]", "", text)
df.loc[:, 'text'] = df['text'].apply(clean_text)
display(df.head())

,text,spam
0,subject naturally irresistible your corporate ...,1
1,subject the stock trading gunslinger fanny is...,1
2,subject unbelievable new homes made easy im w...,1
3,subject color printing special request addit...,1
4,subject do not have money get software cds fr...,1


In [ ]:
import nltk
nltk.download("punkts")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("punkt_tab")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))

[nltk_data] Error loading punkts: Package 'punkts' not found in index
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
df.loc[:,'text']=df['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))
display(df.head())

,text,spam
0,subject naturally irresistible corporate ident...,1
1,subject stock trading gunslinger fanny merrill...,1
2,subject unbelievable new homes made easy im wa...,1
3,subject color printing special request additio...,1
4,subject money get software cds software compat...,1


In [ ]:
print(df['text'][10])

subject las vegas high rise boom las vegas fast becoming major metropolitan city new high rise towers expected built around las vegas strip within next years condominiums boom begun buy first early phase pre construction pricing available las vegas high rises including trump cosmopolitan mgm turnberry icon sky among others join interest list http www verticallv com message sent realty one highrise learn www verticallv comif wish excluded future mailings please reply word remove subject line


In [ ]:
from nltk.stem import PorterStemmer
stemmer=PorterStemmer()
df.loc[:,'text']=df['text'].apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))
df['text'][10]

'subject la vega high rise boom la vega fast becom major metropolitan citi new high rise tower expect built around la vega strip within next year condominium boom begun buy first earli phase pre construct price avail la vega high rise includ trump cosmopolitan mgm turnberri icon sky among other join interest list http www verticallv com messag sent realti one highris learn www verticallv comif wish exclud futur mail pleas repli word remov subject line'

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
X=df['text']
Y=df['spam']
tfidf=TfidfVectorizer(max_features=3000)
X_tfidf=tfidf.fit_transform(X)
X_tfidf.shape

(5695, 3000)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X_tfidf,Y,test_size=0.20,random_state=42)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
model=MultinomialNB()
model.fit(X_train,Y_train)
Y_predict=model.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy_score(Y_test,Y_predict)


0.9798068481123793

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(Y_test,Y_predict)


array([[839,   4],
       [ 19, 277]])

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(Y_test,Y_predict))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       843
           1       0.99      0.94      0.96       296

    accuracy                           0.98      1139
   macro avg       0.98      0.97      0.97      1139
weighted avg       0.98      0.98      0.98      1139



In [ ]:
def predict_message(message):
  # Preprocessing the message
  msg = message.lower()
  msg = re.sub((r'[^a-z\s]'), "", msg)
  token = word_tokenize(msg)
  token = [word for word in token if word not in stop_words]
  tokens = [stemmer.stem(word) for word in token]
  clean_msg = " ".join(tokens)

  # TF-IDF transformation
  vec = tfidf.transform([clean_msg])

  # Prediction
  result_class = model.predict(vec)[0] # 0 for Ham, 1 for Spam
  probabilities = model.predict_proba(vec)[0] # Get probabilities for both classes

  # Map class to label and get its probability
  if result_class == 1:
    label = 'Spam'
    probability = probabilities[1] # Probability of being spam
  else:
    label = 'Ham'
    probability = probabilities[0] # Probability of being ham

  return label, probability

In [ ]:
text_message="Congratulations you won a lottery of 1000000, click the link below to claim your winning prize otherwise it will be given to other so click urgently to claim your money"
predict_message(text_message)

('Spam', np.float64(0.9094083261096753))